# report05 — 파형 검증 — **Sionna 로 대조**

**핵심.** 패시브 레이더가 빌려 쓰는 조명 신호(WiFi·LTE·5G)가 **정말 3GPP/IEEE 규격대로 생성됐는지** 못박는다 — 규격서를 손으로 옮긴 파형을 검증된 라이브러리로 대조해 세 신호 모두 상관 1.0000 로 소수점 한계까지 일치함을 보인다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | 검증된 통신 라이브러리 **Sionna PHY** 는 OFDM 변조 엔진과 3GPP 5G 뉴머롤로지 표는 주지만, WiFi·LTE·5G-SSB 같은 **기회신호 파형을 통째로 만들어 주는 생성기는 없다**(5G 데이터 채널 위주). 그래서 규격서대로 재구성한 파형이 실제로 규격과 일치하는지는 독립 검증이 필요하다. |
| **② 선행 연구의 방식** | Sionna 를 센싱에 쓰는 선행은 조명원 파형을 **규격대로 재구성**해 쓴다 — 5G NR OFDM 패시브 레이더(Wypich&Zielinski, Sensors 2026), LTE450 패시브 레이더(Demissie, IET RSN 2025), 5G SSB 드론 검출(Jopanya&Osorio, SPAWC 2025, arXiv:2504.02641). 파형·채널 계층 자체는 검증된 **Sionna PHY** 를 그대로 신뢰한다. |
| **③ 쓴 라이브러리·결합** | **Sionna PHY** 의 독립 OFDM 변조기(`sionna.phy.ofdm.OFDMModulator`)로 같은 자원격자를 다시 변조해 우리 파형을 채점하고, 5G 뉴머롤로지는 `CarrierConfig`(3GPP TS 38.211 구현)에서 읽어온다 — 라이브러리가 이미 아는 것을 다시 짜지 않는다(중복계산 회피). 파형 합성은 우리 `waveforms.py`, 채점은 Sionna 로 역할이 갈린다. |
| **④ 검증** | 자작↔Sionna 상관 1.0000(WiFi·LTE·5G 모두), NMSE -135.2~-138.3 dB = float32 반올림 바닥(물리 차이 아님). 슬롯 첫 심볼의 긴 CP 조항 하나만 놓쳐도 상관이 0.05 로 붕괴 — 대조가 규격 미세조항까지 예민하다. |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 자작 WiFi/LTE/5G 파형 (검증 대상) | 3GPP TS 36.211 · TS 38.211 · IEEE 802.11ac 를 읽어 `src/waveforms.py` 에 구현 | 🔴 우리 구현 (검증 대상) |
| 교차대조용 독립 OFDM 변조기 | **`sionna.phy.ofdm.OFDMModulator`** — 같은 자원격자를 우리와 무관하게 다시 변조 | 🟢 라이브러리 (채점자) |
| 5G NR 뉴머롤로지 (μ·SCS·슬롯·CP 길이) | **`sionna.phy.nr.CarrierConfig`** — 3GPP 표를 Sionna 에게 물어봤다 | 🟢 라이브러리 (3GPP TS 38.211 구현) |
| 상관·NMSE 대조 숫자 | `src/waveforms_sionna.py` 가 두 파형을 겹쳐 계산 → JSON 에 기록 | 📐 측정 결과 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sionna-phy` | Sionna PHY (`ofdm`/`nr`/`channel`) — OFDM 변복조 · 3GPP 뉴머롤로지 · RT 경로를 신호에 적용 | 🟢 **Sionna 내부** (PyTorch 백엔드, GPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: 이 리포트는 **재측정을 하지 않는다** — 기존 측정 파이프라인이 남긴 JSON·그림을 재배치한다. 노트북 생성은 초 단위. (원본 교차검증 `waveforms_sionna.py` 자체는 GPU 1장·수 초.)

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2

# 교차검증 자체를 다시 돌려 상관·NMSE 를 재계산하려면:
~/.venvs/py312/bin/python src/waveforms_sionna.py   # 자작 <-> Sionna 대조

# 이 리포트의 숫자·그림은 기존 측정 파이프라인이 남긴 JSON·그림을 재사용한다(재측정 없음).
~/.venvs/py312/bin/python src/make_notebook05.py    # JSON -> report05.ipynb
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report2_waveform_rcs.json` | `crosscheck` 블록 = **이 리포트의 모든 대조 숫자**(상관·NMSE·CP 길이) |
| `outputs/figures/report2_crosscheck.png` | §4·§5 교차검증 — 시간파형·스펙트럼 포갬 + 첫-심볼 CP 민감도 |
| `outputs/figures/report2_sionna_waveforms.png` | §3·§4 Sionna = OFDM 엔진이자 채점자, 잔차 = float32 반올림 |
| `outputs/figures/report2_numerology.png` | §3 5G 뉴머롤로지 — 우리가 안 짜고 CarrierConfig 에서 읽어온 표 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **이 대조가 보증하는 것은 'OFDM 변조 수학'과 '5G 뉴머롤로지' 다.** 즉 IFFT·가드·DC널·심볼별 CP 삽입이 규격대로인가, 그리고 격자 치수(RB·심볼수·CP 길이)가 3GPP 표와 맞는가. 이건 두 독립 구현이 소수점 한계까지 일치함으로 강하게 보증된다.
- **파일럿(기준신호) 배치 위치까지 Sionna 가 독립적으로 확인해 주지는 못한다.** Sionna PHY 에는 WiFi VHT-LTF·LTE CRS·5G SSB **생성기가 없기** 때문이다. 그 배치 좌표는 우리가 3GPP/IEEE 스펙을 읽어 넣은 것이고, 근거는 코드 주석과 `docs/` 에 남겼다. 대조는 그 격자를 **신호로 바꾸는 단계**를 검증한다.
- **여기서 검증하는 것은 '규격 일치' 이지 '탐지 성능' 이 아니다.** 이 파형으로 표적을 얼마나 잘 보는가(거리·속도 분해능)는 앞 리포트(→ report04), 표적이 얼마나 밝은가는 → report06 소관이다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| [report04](report04.ipynb) — 조명원 · 파형 (WiFi/LTE/5G) | **앞 리포트.** 그 파형이 **무엇을 주는가**(대역·PRF·분해능)를 다뤘다. 여기서는 그 파형이 **정말 규격대로인가**를 검증한다 |
| **report05 (여기)** — 파형 검증 | 자작 파형 ↔ Sionna OFDMModulator 교차대조. 상관 1.0000 = 규격 일치의 강한 증거 |
| [report06](report06.ipynb) — 드론이 레이더에 얼마나 밝은가 (RCS) | **다음 리포트.** 조명(파형)을 확정했으니, 이제 표적 쪽 — 드론의 되비침 밝기(RCS)로 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **규격 / 표준(spec)** | 통신 신호가 어떤 칸에 무엇을 담아야 하는지 정한 문서(3GPP·IEEE). 우리는 이걸 읽고 신호를 손으로 만들었다 |
| **OFDM** | 부반송파 수백~수천 개에 데이터를 잘게 나눠 싣는 변조 방식. 이 신호를 만드는 핵심 연산이 IFFT(역푸리에변환)와 CP 삽입 |
| **자원격자(resource grid)** | 시간=가로·주파수=세로의 모눈종이. OFDM 신호는 이 칸을 채워 만든다 |
| **순환전치 CP(cyclic prefix)** | 각 OFDM 심볼 **끝부분을 복사해 앞에 덧붙이는 완충 구간**. 다중경로로 신호가 번져도 심볼끼리 안 겹치게 막는 보호띠 |
| **첫 심볼 긴 CP** | 3GPP 는 한 슬롯의 **첫 심볼에만 CP 를 더 길게** 준다(정렬을 맞추려고). LTE·5G 의 미세하지만 반드시 지켜야 하는 규칙 |
| **상관(correlation)** | 두 신호가 얼마나 똑같은가를 재는 값. **1이면 완전히 같음**, 0이면 무관 |
| **NMSE** | 정규화 평균제곱오차. 두 신호 차이의 크기를 dB 로 표시 — **더 음수(작을수록)일수록 완벽히 같다.** -135 dB 는 사실상 0 |
| **float32** | 컴퓨터가 소수를 32비트로 저장하는 방식. 유효숫자 약 7자리에서 반올림한다 → 두 계산이 완벽히 같아도 마지막 자리에 ~-140 dB 수준의 미세한 차이가 남는다 |
| **뉴머롤로지** | SCS(부반송파 간격)·슬롯·심볼수·CP 길이 등 5G 물리계층 격자 파라미터 |
| **Sionna PHY** | 검증된 오픈소스 물리계층 라이브러리. OFDM 변복조·3GPP 뉴머롤로지를 제공. 여기선 우리 파형을 채점하는 **독립 참조**로 쓴다 |
| **OFDMModulator** | Sionna 의 OFDM 변조기 — 자원격자를 받아 IFFT·CP 삽입으로 시간파형을 만든다 |
| **CarrierConfig** | Sionna 가 3GPP 5G NR 뉴머롤로지 표를 담고 있는 객체. 우리가 표를 안 짜도 됨 |

</details>

---


## §1. Sionna 의 공백 — 기회신호 파형 생성기가 없다

레이더가 되돌아온 신호에서 표적을 찾으려면 **원래 나간 신호가 정확히 어떻게 생겼는지**를 알아야 한다(그래야 '되돌아온 것' 과 '나간 것' 을 상관으로 맞춰볼 수 있다). 패시브 레이더는 WiFi·LTE·5G 같은 **기회신호(이미 공중에 떠 있는 통신 신호)** 를 조명원으로 빌려 쓰므로, 그 세 파형을 3GPP·IEEE 규격대로 재구성해 둔다:

- **WiFi 802.11ac** — IEEE 802.11ac 규격
- **LTE Rel-9** — 3GPP TS 36.211
- **5G NR Rel-16** — 3GPP TS 38.211

검증된 통신 물리계층 라이브러리 **Sionna PHY** 는 OFDM 변조 엔진과 3GPP 5G 뉴머롤로지 표는 제공하지만, 이런 **기회신호 파형을 통째로 만들어 주는 생성기는 없다**(5G 데이터 채널 위주). 그래서 파형은 우리 `waveforms.py` 가 규격을 따라 직접 합성한다.

각 신호는 **자원격자(시간=가로·주파수=세로의 모눈종이)** 의 칸마다 무엇을 켤지 규격이 정해두고 있다. 그 칸을 규격대로 채운 뒤, **OFDM(부반송파 수천 개에 데이터를 잘게 나눠 싣는 변조 방식)** 로 시간축 파형을 만든다.

**규격서를 옮기는 단계에는 실수가 숨을 수 있다.** 부반송파 하나를 잘못 배치하거나 심볼 앞에 붙이는 **순환전치(CP — 심볼 끝을 복사해 앞에 덧붙이는 완충 구간)** 의 길이를 헷갈려도, 만든 파형만 눈으로 봐서는 티가 나지 않는다. 그래서 이 단계가 규격과 맞는지는 **눈이 아니라 검증된 라이브러리로** 확인한다 — 우리와 무관한 도구가 같은 격자를 다시 변조해, 결과가 포개지는지 본다(§3). Sionna PHY 에 파형 생성기가 없다는 이 공백을, 선행 연구는 어떻게 다뤘는지부터 본다(§2).

---
## §2. 선행 연구는 이 공백을 어떻게 다뤘나 — 규격대로 재구성

조명원 파형을 규격대로 재구성해 쓰는 것은 이 분야의 표준 관행이다. Sionna 를 센싱에 쓰는 선행도 파형 생성기가 없는 이 공백을 같은 방식으로 메운다:

| 선행 | 조명원 | 규격대로 재구성한 것 |
|---|---|---|
| Wypich&Zielinski, Sensors 2026 | 5G NR OFDM | PSS/SSS·DM-RS·PDSCH |
| Demissie, IET RSN 2025 | LTE450 | LTE OFDM 기준요소 |
| Jopanya&Osorio, SPAWC 2025 (arXiv:2504.02641) | 5G SSB | SSB 블록 |

공통점은 둘이다. **①** 파형은 규격서(3GPP·IEEE)를 읽어 재구성하고, **②** 파형·채널 계층 자체는 검증된 **Sionna PHY** 를 그대로 신뢰한다. 그리고 재구성한 파형이 규격과 맞는지는 검증된 라이브러리로 대조해 확인한다. 우리도 같은 길을 따른다 — 파형은 우리가 규격대로 채우고(§1), 그 결과를 Sionna PHY 로 채점한다(§3).

---
## §3. 우리가 쓴 방식 — 같은 격자, 두 변조기

**Sionna PHY** 는 널리 쓰이는 검증된 통신 물리계층 라이브러리입니다(NVIDIA, Apache-2.0). 파형이 규격대로인지는 이 라이브러리를 기준 삼아 확인합니다:

1. **규격대로 채운 자원격자를 준비한다.**
2. **그 격자를 Sionna PHY 의 독립 OFDM 변조기(`sionna.phy.ofdm.OFDMModulator`)에 넣어 다시 변조한다** — 파형 합성 코드와 한 줄도 겹치지 않는 별개 엔진이다.
3. **두 시간파형을 겹쳐본다.** 격자를 규격대로 변조했다면 결과는 같아야 한다.

검증된 라이브러리가 같은 격자에서 **같은 파형**을 내면, 파형이 규격대로라는 강한 증거가 됩니다.

### 뉴머롤로지도 Sionna 에서 읽어온다

5G 는 격자의 치수(부반송파 간격·슬롯당 심볼수·CP 길이)를 **뉴머롤로지** 라는 규격 표로 정해둡니다. 이 표는 Sionna 의 `CarrierConfig`(3GPP TS 38.211 구현)에서 **그대로 읽어옵니다** — 라이브러리가 이미 아는 걸 다시 짜지 않는다는 원칙입니다.

![numerology](outputs/figures/report2_numerology.png)

*(위 표의 μ·심볼수·슬롯수·CP 유형은 전부 `CarrierConfig(subcarrier_spacing, n_size_grid)` 에서 그대로 읽어온 값입니다.)*

### 이 대조가 보증하는 것과 못 하는 것

Sionna PHY 에는 WiFi·LTE·5G-SSB 파형 생성기가 없으므로(§1), 이 대조는 파형 전체를 라이브러리로 대체하는 것이 아니라 **격자를 신호로 바꾸는 OFDM 변조 단계**를 라이브러리로 검증하는 것입니다. 경계를 분명히 해둡니다:

| 이 대조가 **확인해 주는 것** | 이 대조가 **못 하는 것** |
|---|---|
| OFDM 변조 수학(IFFT·가드·DC널·심볼별 CP)이 규격대로인가 | 파일럿(CRS/SSB/VHT-LTF) **위치**의 독립 확인 |
| 5G 뉴머롤로지(RB·심볼수·CP 길이)가 3GPP 표와 맞는가 | (Sionna 에 그 생성기가 없어 위치는 규격을 읽어 넣음) |

파일럿을 어느 칸에 놓느냐는 3GPP/IEEE 규격을 읽어 넣은 것이고, 근거는 코드 주석과 `docs/` 에 남겼습니다. 대조는 그 격자를 **신호로 바꾸는 단계**를 소수점 한계까지 검증합니다.

---
## §4. 검증 결과 — 상관 1.0000, 남은 차이는 반올림뿐

같은 자원격자를 Sionna PHY 의 독립 OFDM 변조기로 다시 변조해 겹쳤습니다. **세 신호 모두 사실상 완벽히 일치**합니다:

| 표준 | 표본 수 | 표본율 $f_s$ | **상관** (1=동일) | **NMSE** (작을수록 동일) |
|---|---|---|---|---|
| WiFi 802.11ac | 4,160 | 80.00 MHz | **1.0000** | **-138.3 dB** |
| LTE Rel-9 | 30,720 | 30.72 MHz | **1.0000** | **-135.6 dB** |
| 5G NR Rel-16 | 61,440 | 122.88 MHz | **1.0000** | **-135.2 dB** |

**읽는 법 — 상관.** 두 신호가 얼마나 같은가를 재는 값입니다. 1이면 완전히 같고, 0이면 전혀 다릅니다. 세 신호 모두 소수 넷째 자리까지 **1.0000** — 두 신호를 겹쳐 그리면 한 줄로 포개집니다.

**읽는 법 — NMSE.** 두 신호의 '차이의 크기' 를 dB 로 나타낸 값으로, **더 음수일수록(작을수록) 완벽히 같다**는 뜻입니다. 여기서 -135.2 ~ -138.3 dB 는 상상하기 어려울 만큼 작습니다 — 신호 세기 대비 차이가 약 $10^{-13.5}$ 배(≈$3{\times}10^{-14}$)라는 뜻이니까요.

### 그럼 그 -135 dB 마저 왜 0이 아닌가? — float32 반올림

**남은 미세한 차이는 물리가 아니라 컴퓨터 산수의 한계입니다.** 컴퓨터는 소수를 **float32**(32비트) 로 저장하는데, 유효숫자 약 7자리에서 반올림합니다. 그래서 두 계산이 수학적으로 완전히 같아도, 서로 다른 순서로 더하고 곱하면 **마지막 자리에 ~-140 dB 수준의 티끌**이 남습니다. 우리가 본 -135 dB 는 바로 그 **반올림 바닥(floor)** 에 붙어 있습니다.

> **비유 —** 같은 계산을 계산기 두 대로 하면 화면엔 같은 답이 뜨지만, 내부적으로 마지막 자릿수만 아주 미세하게 다를 수 있습니다. 그 차이는 '두 계산이 틀렸다' 가 아니라 '계산기가 소수를 딱 그만큼만 정밀하게 다룬다' 는 뜻입니다.

아래 그림이 이걸 한눈에 보여줍니다. 시간파형(위)·스펙트럼(가운데)이 두 색으로 완전히 포개지고, 잔차(오른쪽 아래 패널)는 -135 dB 언저리의 평평한 잡음 — 즉 **float32 반올림** 입니다.

![sionna waveforms](outputs/figures/report2_sionna_waveforms.png)

![.](outputs/renders/anim/spectrum_wifi.gif)

<sub>WiFi 파형 스펙트럼(넓은 대역) — 시간파형·스펙트럼 모두 Sionna 와 상관 1.0000 로 일치했다.</sub>

In [ ]:
# §4 재현 — 자작 파형과 Sionna 파형을 겹쳐 계산한 상관·NMSE 를 그대로 읽는다 (하드코딩 없음)
import json
J = json.load(open('outputs/report2_waveform_rcs.json'))
for k in ('wifi', 'lte', 'nr'):
    d = J['crosscheck'][k]
    print(f"{d['name']:14s}  표본={d['n']:>7,}  "
          f"상관={d['corr']:.4f}  NMSE={d['nmse_db']:7.1f} dB")

# 대조 자체를 처음부터 다시 돌리려면:  python src/waveforms_sionna.py

---
## §5. 검증의 예민함 — 슬롯 첫 심볼의 긴 CP

상관이 1.0000 이라는 건 그냥 '대충 비슷하다' 가 아니라 **규격의 미세한 부분까지 전부 맞았다**는 뜻입니다. 그걸 잘 보여주는 조항이 **순환전치(CP)의 길이 규칙** 입니다.

**CP 는 각 OFDM 심볼의 끝부분을 복사해 앞에 덧붙인 완충 구간**입니다(다중경로로 신호가 번져도 심볼끼리 안 겹치게 막는 보호띠). 그런데 3GPP 는 한 슬롯의 **첫 심볼에만 이 완충을 조금 더 길게** 주라고 정해뒀습니다 — 슬롯 경계를 정렬하려는 목적입니다. 측정된 CP 길이를 보면 규칙이 그대로 드러납니다:

| 표준 | 앞쪽 심볼들의 CP 길이 (샘플) | CP 가 균일한가 |
|---|---|---|
| WiFi 802.11ac | [64] | ✅ 균일 (첫 심볼도 같음) |
| LTE Rel-9 | [160, 144, 144, 144, …] | ❌ **첫 칸만 160, 이후 144** |
| 5G NR Rel-16 | [352, 288, 288, 288, …] | ❌ **첫 칸만 352, 이후 288** |

**이 규칙 하나가 얼마나 중요한지**, 대조의 예민함으로 확인할 수 있습니다. 만약 이 '첫 심볼 긴 CP' 를 놓치고 **모든 심볼에 같은 CP** 를 줬다면, 두 번째 심볼부터 시간축이 어긋나 파형 전체가 밀립니다. 그 결과 상관은:

| 표준 | CP 규칙을 지켰을 때 | 첫-심볼 긴 CP 를 놓쳤을 때 |
|---|---|---|
| WiFi 802.11ac (원래 균일) | 1.0000 | 1.0000 *(변화 없음 — CP 가 원래 균일해서)* |
| LTE Rel-9 | 1.0000 | **0.0634** ← 무너짐 |
| 5G NR Rel-16 | 1.0000 | **0.0451** ← 무너짐 |

LTE·5G 는 상관이 1.0000 에서 **0.06·0.05 로 폭락**합니다 — 즉 이 대조는 **CP 배치 규칙 하나까지 잡아낼 만큼 예민**하다는 뜻입니다. (WiFi 는 원래 CP 가 균일해서 이 조항이 없고, 그래서 값이 안 변합니다 — 한 신호만 봐서는 걸러낼 수 없을 종류의 미세한 차이가, **여러 표준을 함께 대조**하기에 드러납니다.)

![crosscheck](outputs/figures/report2_crosscheck.png)

> **정리 —** 상관 1.0000 은 '눈으로 봐서 비슷하다' 와는 차원이 다른 성적표입니다. 슬롯 첫 심볼의 CP 를 한 조각만 틀려도 0.05 로 무너지는 시험을, 세 신호가 전부 **만점**으로 통과했습니다. 규격대로 재구성한 파형이 검증된 라이브러리 Sionna PHY 와 **사실상 완전히 일치합니다.**

---
> **다음 리포트**: report06 — 조명(파형)이 규격대로임을 확인했으니, 이제 **표적 쪽** 으로 갑니다. 드론이 레이더 전파를 **얼마나 밝게 되비치는가**(RCS — 레이더 되비침 밝기)를 다룹니다. 표적이 밝아야 잡히니, 이건 탐지의 다른 절반입니다.